# AAF to Neptune Graph

Comprehensive parser - filters nodes/edges with N/A IDs

In [1]:
import gzip, json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union
from boto3 import Session
from neptune_graph_manager.types import Edge, Node, NodeArray
from neptune_graph_manager import GraphBuilder, NeptuneGraphManager
from dotenv import load_dotenv
import os
from ast import literal_eval

load_dotenv()

/Users/crisleoo/workplace/ThreatForest-internal/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
session = Session(**literal_eval(os.getenv("SESSION_PARAMS", {})))
neptune_manager = NeptuneGraphManager(session=session, graph_id="g-csewvv20d7", embedding_model="cisco-ai/SecureBERT2.0-biencoder")
summary = neptune_manager.get_summary()

with open('data/graph_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Graph summary saved to data/graph_summary.json")

📈 Graph summary generated successfully!
Graph summary saved to data/graph_summary.json


In [4]:
query = """
MATCH p=()-[]-()
RETURN p
"""
neptune_manager.query_ops.execute_query(query=query, visualize=True, interactive=True, max_nodes=10)

ℹ️  Limiting visualization to top 10 nodes by degree.
✅ Interactive graph saved to: graph.html
📊 Graph contains 10 nodes and 22 edges

📌 How to use the interactive graph:
   • Hover over nodes to see full details
   • Click and drag nodes to rearrange the layout
   • Use mouse wheel or buttons to zoom in/out
   • Use navigation buttons for panning


[{'p': [{'~id': 'attack-pattern--9123cd1a-9c09-4594-a9c4-76eee557dca2',
    '~entityType': 'node',
    '~labels': ['Technique'],
    '~properties': {'stix_id': 'attack-pattern--9123cd1a-9c09-4594-a9c4-76eee557dca2',
     'aws_api_events': 'ec2:AuthorizeSecurityGroupIngress',
     'created': '2020-06-24T16:55:46.243Z',
     'description': 'Adversaries may disable or modify a firewall within a cloud environment to bypass controls that limit access to cloud resources. Cloud firewalls are separate from system firewalls that are described in Disable or Modify System Firewall.\n\nCloud environments typicall utilize restrictive security groups and firewall rules that only allow network activity from trusted IP addresses via expected ports and protocols. An adversary may introduce new firewall rules or policies to allow access into a victim cloud environment and/or move laterally from the cloud control plane to the data plane. For example, an adversary may use a script or utility that creates 

In [26]:
query_embedding = neptune_manager.embedding_ops.get_embedding("Perform an attack to avoid possible detection of tools and activities")

neptune_query = f"""
CALL neptune.algo.vectors.topKByEmbedding(
  {query_embedding}
)
YIELD node, score
RETURN node, score
"""

results = neptune_manager.query_ops.execute_query(neptune_query)
node_results = NodeArray().from_neptune_nodes(results)

for n in node_results:
    print(f"{n.properties['name']} : {n.properties['description'][:200]}")

Software Discovery : Adversaries may attempt to get a listing of software and software versions that are installed on a system or in a cloud environment. Adversaries may use the information from [Software Discovery](https
Log Enumeration : Adversaries may enumerate system and service logs to find useful data. These logs may highlight various types of valuable insights for an adversary, such as user authentication records ([Account Disco
Hide Artifacts : Adversaries may attempt to hide artifacts associated with their behaviors to evade detection. Operating systems may have features to hide various artifacts, such as important system files and administ
Create Cloud Instance : An adversary may create a new instance or virtual machine (VM) within the compute service of a cloud account to evade defenses. Creating a new instance may allow an adversary to bypass firewall rules 
Network Sniffing : Adversaries may passively sniff network traffic to capture information about an environment, incl

In [ ]:
node_results[0].get

'attack-pattern--840bab84-6944-4746-a785-62b1cda64740'

In [21]:
query = f"""
MATCH( n {{`~id`: '{node_results[0].id.id}'}} )
CALL neptune.algo.vectors.get(n)
YIELD embedding
RETURN embedding
"""
embed_results = neptune_manager.query_ops.execute_query(query)

In [25]:
len(embed_results[0]['embedding'])

768